# 00｜先把資料看懂：資料結構與對齊方式

## 專案名稱

**半導體蝕刻製程智慧分析與虛擬量測**  
Semiconductor Etch Process Intelligence & Virtual Metrology

---

## 這份 Notebook 要做什麼？

在真的開始清資料、看製程趨勢或做模型以前，我想先把最基本的問題弄清楚：**這份資料到底有哪些檔案、每一列代表什麼、Lot / Wafer / Measurement 之間怎麼對上。**

這一份只做「資料理解」。我不在這裡補值、不刪資料、不做特徵工程，也不訓練模型，避免一開始就把資料結構和後面的分析混在一起。

我主要想確認：

1. 專案用了哪些原始檔案。
2. 89-point measurement 實際有多少 Lot、Wafer 和 measurement records。
3. 每片有量測資料的 Wafer 是否真的都有 89 筆 records。
4. `Lot_status.xlsx` 一共記錄多少 Wafer。
5. `Process_data.nc` 裡一片 Wafer 的時間序列是怎麼存的。
6. 不同 Wafer 的 Process Feature 欄位是否一致。
7. 哪些 Process Features 可以跨全部 Wafer 一起使用。
8. Process Data 和 Measurement 能不能靠同一個 Wafer key 精準對齊。
9. 後面分析時，什麼時候應該看 Lot、Wafer，什麼時候才看 measurement point。

這一步先把地基打好，後面 01 才有辦法針對真正的資料品質問題處理。


## 1. 先了解這份資料集在做什麼

這個專案用的是 BOSCH Plasma-Etching 公開資料集。這一階段我先集中在三種資料：

- Process Parameters
- 89-point Wafer Measurements
- Lot / Conditioning Metadata

資料集說明裡提到，Process Parameters 大約以 **5 Hz** 記錄，所以一片 Wafer 會有一段時間序列；89-point Measurement 則是在製程後，量 Wafer 上不同位置的結果。

這一版我先不碰比較大的 OES 光譜資料，因為我想先把最基本的關係弄清楚：

**Process Data → Wafer → 最後的 Measurement Result**

等這條線確認沒問題，後面再看 Quality、Sequence Effect、Process Drift 和 Virtual Metrology。

另外，這裡不先假設 Conditioning 一定會影響品質或製程，那要留到後面用資料驗證。


## 2. 載入套件與設定專案路徑

先把讀 CSV、Excel、NetCDF 和資料處理會用到的套件載入。這一步只是準備環境，還沒有真的讀資料，也不會改到任何 Raw Data。


In [1]:
# 基本資料處理
from pathlib import Path

import numpy as np
import pandas as pd

# NetCDF 資料
from netCDF4 import Dataset

# 顯示設定
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print("套件載入完成")

套件載入完成


**這個結果代表什麼？**

顯示「套件載入完成」代表目前 Python 環境可以正常使用這些套件，後面才能繼續讀 CSV、Excel 和 NetCDF。

如果這一格就出現 ImportError，應該先處理環境問題，而不是直接往下分析。


### 2.1 找到專案根目錄

我用 `Path` 來處理路徑，避免把某一台電腦的完整路徑直接寫死在程式裡。

如果 Notebook 是從 `notebooks/` 執行，就往上一層找專案根目錄；如果現在本來就在專案根目錄，就直接用目前位置。這樣之後搬到別台電腦或放到 GitHub 會比較好維護。


In [2]:
# 取得目前 Python / Jupyter 的工作目錄
CURRENT_DIR = Path.cwd()

# 如果目前工作目錄剛好是 notebooks，
# 就往上一層找專案根目錄；
# 否則目前工作目錄本身就是專案根目錄。
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print("目前工作目錄：", CURRENT_DIR)
print("專案根目錄：", PROJECT_ROOT)
print("原始資料目錄：", RAW_DATA_DIR)

目前工作目錄： C:\Users\momo8\Desktop\semiconductor-etch-process-intelligence\notebooks
專案根目錄： C:\Users\momo8\Desktop\semiconductor-etch-process-intelligence
原始資料目錄： C:\Users\momo8\Desktop\semiconductor-etch-process-intelligence\data\raw


**這個結果代表什麼？**

目前工作目錄和專案根目錄都指向 `semiconductor-etch-process-intelligence`，Raw Data 也正確指到 `data/raw/`。

所以這份 Notebook 現在讀的就是這個專案底下的資料，不是其他資料夾裡的同名檔案。


### 2.2 先確認原始檔案都有找到

正式開始讀資料前，我先檢查這次分析需要的原始檔案是不是都存在。

如果其中任何一個顯示「找不到」，後面的數字就不值得繼續看，因為代表輸入資料根本不完整。


In [3]:
process_path = RAW_DATA_DIR / "Process_data.nc"
dictionary_path = RAW_DATA_DIR / "Dictionary_process.nc"
lot_status_path = RAW_DATA_DIR / "Lot_status.xlsx"
measurement_path = RAW_DATA_DIR / "Si_Oxide_etch_89_points.csv"
readme_path = RAW_DATA_DIR / "Readme.pdf"
wafer_layout_path = RAW_DATA_DIR / "Wafer_layout.pdf"

raw_files = {
    "Process Data": process_path,
    "Process Dictionary": dictionary_path,
    "Lot Status": lot_status_path,
    "89-point Measurement": measurement_path,
    "Readme": readme_path,
    "Wafer Layout": wafer_layout_path,
}

for name, path in raw_files.items():
    print(f"{name:<25} {'存在' if path.exists() else '找不到'}")

Process Data              存在
Process Dictionary        存在
Lot Status                存在
89-point Measurement      存在
Readme                    存在
Wafer Layout              存在


**這個結果代表什麼？**

6 個需要用到的原始檔案全部顯示「存在」。

所以目前輸入資料是完整的，可以繼續往下確認資料結構。


### 2.3 這個專案目前會用到哪些原始檔？

| 檔案 | 我會拿它做什麼 |
|---|---|
| `Process_data.nc` | 每片 Wafer 的設備製程時間序列 |
| `Dictionary_process.nc` | 把 `Process_data.nc` 裡的編碼數值解回真正的數值 |
| `Lot_status.xlsx` | Lot、日期、Wafer 數量、Conditioning 和 NaN 紀錄 |
| `Si_Oxide_etch_89_points.csv` | 每片 Wafer 的 89-point 空間量測結果 |
| `Readme.pdf` | 資料集和實驗設計說明 |
| `Wafer_layout.pdf` | 200 mm Wafer 的量測位置配置 |

這一份先只搞清楚資料結構，不急著把任何一個欄位丟進模型。


## 3. 先看 89-point Measurement 長什麼樣

我先從 `Si_Oxide_etch_89_points.csv` 開始，因為它代表製程完成後每片 Wafer 的空間量測結果。

第一步只看資料有幾列、幾欄，以及前幾筆資料長什麼樣，先確認「一列到底代表什麼」。


In [4]:
measurement_df = pd.read_csv(measurement_path)

print("資料維度：", measurement_df.shape)

measurement_df.head()

資料維度： (7832, 11)


,experiment_key,lot_number,wafer_number,X,Y,preox_thickness,postox_thickness,postox_thickness_nan,stepheight,oxide_etch,si_etch
0,2024-07-02_01,1,1,-19000,-95000,1.002226,0.5259,0.5259,52.9890,0.476326,52.4631
1,2024-07-02_01,1,1,0,-95000,1.003400,0.5408,0.5408,52.5790,0.462600,52.0382
2,2024-07-02_01,1,1,19000,-95000,1.002493,0.5240,0.5240,53.2003,0.478493,52.6763
3,2024-07-02_01,1,1,-57000,-76000,0.999300,0.5420,0.5420,52.8116,0.457300,52.2696
4,2024-07-02_01,1,1,-38000,-76000,1.000041,0.4795,0.4795,46.8974,0.520541,46.4179


**這個結果代表什麼？**

這份 CSV 是 **7,832 rows × 11 columns**。

從前幾筆可以看到，同一個 `experiment_key` 會重複出現，但 X/Y 位置不同。所以一列不是一片 Wafer，而是：

> **一片 Wafer 上的一個量測位置。**

這個差別很重要，因為後面做 Wafer-level 分析時不能直接把 7,832 列當成 7,832 個獨立樣本。


### 3.1 先看欄位型態和缺失概況

我用 `DataFrame.info()` 看每個欄位的資料型態和 non-null count。

這裡的目的只是先找出「哪個欄位看起來有缺失」，真正要不要補、怎麼處理，留到 01 再做。


In [5]:
measurement_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7832 entries, 0 to 7831
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   experiment_key        7832 non-null   str    
 1   lot_number            7832 non-null   int64  
 2   wafer_number          7832 non-null   int64  
 3   X                     7832 non-null   int64  
 4   Y                     7832 non-null   int64  
 5   preox_thickness       7832 non-null   float64
 6   postox_thickness      7832 non-null   float64
 7   postox_thickness_nan  7675 non-null   float64
 8   stepheight            7832 non-null   float64
 9   oxide_etch            7832 non-null   float64
 10  si_etch               7832 non-null   float64
dtypes: float64(6), int64(4), str(1)
memory usage: 673.2 KB


**這個結果代表什麼？**

11 個欄位裡，只有 `postox_thickness_nan` 的 non-null count 少於 7,832，其他欄位目前都是完整的。

所以 Missing Value 問題不是整份 Measurement Data 到處都有，而是集中在這一個欄位。這裡先記下來，不補值也不刪資料，01 再確認它和 `postox_thickness` 到底是什麼關係。


### 3.2 主要欄位先用白話整理

| 欄位 | 這裡怎麼理解 |
|---|---|
| `experiment_key` | Wafer 識別碼，例如 `2024-07-02_01` |
| `lot_number` | Lot 編號 |
| `wafer_number` | 這片 Wafer 在該 Lot 裡的編號 |
| `X`, `Y` | Wafer 上的量測座標 |
| `preox_thickness` | 製程前 SiO₂ 厚度 |
| `postox_thickness` | 製程後 SiO₂ 厚度，資料集提供的完整版本 |
| `postox_thickness_nan` | 保留原始 post-etch fitting failure 的版本 |
| `stepheight` | 製程後 Step Height |
| `oxide_etch` | 算出來的 SiO₂ Etch Depth |
| `si_etch` | 算出來的 Silicon Etch Depth |

這裡要先記住一件事：`oxide_etch` 和 `si_etch` 不是完全獨立的原始量測，它們是由其他欄位算出來的。

所以以後如果把 `si_etch` 當 prediction target，就不能又把能直接算出 `si_etch` 的 metrology 欄位一起當 predictor，不然模型等於提前看到答案，會造成 target leakage。


### 3.3 7,832 列到底代表多少 Lot、多少 Wafer？

7,832 是 measurement-point 的列數，不是 Wafer 數。

所以這裡我做兩件事：

1. 算出 Lot、Wafer 和 `experiment_key` 各有多少 unique values。
2. 確認 `experiment_key` 和 `lot_number + wafer_number` 真的一對一對得起來。

第二點不能只看「兩邊剛好都是 88 個」就當作沒問題，因為 unique count 一樣不代表 key 一定配對正確。

後面如果做模型，分析單位會是一片 Wafer；但同一個 Lot 裡的 Wafer 共享相同的實驗背景，所以模型驗證時不能把 88 片當成彼此完全獨立的樣本。


In [6]:
# 建立唯一的 Wafer ↔ experiment_key 對應表
n_rows = len(measurement_df)
n_lots = measurement_df["lot_number"].nunique()

wafer_key_map = (
    measurement_df[
        ["experiment_key", "lot_number", "wafer_number"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

n_wafers = (
    wafer_key_map[
        ["lot_number", "wafer_number"]
    ]
    .drop_duplicates()
    .shape[0]
)

n_experiments = wafer_key_map["experiment_key"].nunique()

# 一個 experiment_key 只能對應一片 Lot/Wafer
experiment_key_to_one_wafer = (
    wafer_key_map["experiment_key"]
    .duplicated()
    .sum()
    == 0
)

# 一片 Lot/Wafer 也只能對應一個 experiment_key
wafer_to_one_experiment_key = (
    wafer_key_map[
        ["lot_number", "wafer_number"]
    ]
    .duplicated()
    .sum()
    == 0
)

print(f"量測資料列數：{n_rows:,}")
print(f"Lot 數量：{n_lots}")
print(f"Wafer 數量：{n_wafers}")
print(f"Experiment Key 數量：{n_experiments}")
print(
    "每個 experiment_key 是否只對應一片 Wafer：",
    experiment_key_to_one_wafer,
)
print(
    "每片 Wafer 是否只對應一個 experiment_key：",
    wafer_to_one_experiment_key,
)


量測資料列數：7,832
Lot 數量：10
Wafer 數量：88
Experiment Key 數量：88
每個 experiment_key 是否只對應一片 Wafer： True
每片 Wafer 是否只對應一個 experiment_key： True


**這個結果代表什麼？**

結果可以直接確認：

- Measurement records：**7,832**
- Lot：**10**
- 有 Measurement 的 Wafer：**88**
- `experiment_key`：**88 個**
- 每個 `experiment_key` 只對應一個 Lot/Wafer：`True`
- 每個 Lot/Wafer 也只對應一個 `experiment_key`：`True`

所以 `experiment_key` 可以放心拿來當後面跨檔案對齊的 Wafer key。

另外，之後 supervised learning 最多只有 **88 個 Wafer-level samples**，而且它們分在 10 個 Lots 裡，所以模型驗證一定要保留 Lot grouping，不能把 88 片完全打散當成獨立樣本。


### 3.4 每片 Wafer 是否真的都有 89 筆 Measurement Records？

這裡先按照 `experiment_key + lot_number + wafer_number` 分組，只看每片 Wafer 有幾列。

要注意，**89 rows 還不能直接等於 89 個 unique spatial positions**。有可能理論上有重複 X/Y，所以 01 還會另外檢查 duplicate key 和 spatial grid。


In [7]:
# 每片 Wafer 的 measurement record 數量
records_per_wafer = (
    measurement_df
    .groupby(
        ["experiment_key", "lot_number", "wafer_number"]
    )
    .size()
    .reset_index(name="measurement_records")
)

records_per_wafer["measurement_records"].describe()


count    88.0
mean     89.0
std       0.0
min      89.0
25%      89.0
50%      89.0
75%      89.0
max      89.0
Name: measurement_records, dtype: float64

**這個結果代表什麼？**

88 片 Wafer 的 row count 完全一樣：

- 平均 = **89**
- 最小 = **89**
- 最大 = **89**
- 標準差 = **0**

所以目前可以確定每片 measured wafer 都有 **89 筆 records**。

但這一格只證明 row 數一致，還沒有證明 X/Y 真的是 89 個不重複的位置，這件事留到 01 再驗證。


描述統計看起來都是 89，但我再直接用布林條件檢查一次「是不是每一片都剛好等於 89」，避免只靠平均值和 min/max 下結論。


In [8]:
all_have_89_records = (
    records_per_wafer["measurement_records"]
    .eq(89)
    .all()
)

print(
    "每片 Wafer 是否都有 89 筆 Measurement Records：",
    all_have_89_records,
)


每片 Wafer 是否都有 89 筆 Measurement Records： True


**這個結果代表什麼？**

結果是 `True`，所以 88 片 Wafer 每片都真的有 89 筆 records。

因此：

**88 wafers × 89 records = 7,832 rows**

和 CSV 的總列數完全對得上。

到這裡可以說 row structure 沒問題，但還是不能提前把它寫成「89 個 unique positions」，那要等 01 的 X/Y 檢查通過才成立。


### 3.5 各 Lot 有幾片 Wafer 出現在 89-point Measurement？

前面已經知道每片 measured wafer 都有 89 筆 records，接著我想看每個 Lot 到底有幾片 Wafer 出現在這份 CSV 裡。

這裡先只描述 coverage，不去猜為什麼某些 Lot 少了幾片。


In [9]:
#看各 Lot 到底有幾片 Wafer
wafers_per_lot = (
    measurement_df
    .groupby("lot_number")["wafer_number"]
    .nunique()
    .rename("wafers_with_89_point_measurement")
    .reset_index()
)

wafers_per_lot

,lot_number,wafers_with_89_point_measurement
0,1,9
1,2,10
2,3,10
3,4,10
4,5,10
5,6,10
6,7,6
7,8,10
8,9,9
9,10,4


**這個結果代表什麼？**

各 Lot 在 89-point CSV 裡的 Wafer 數量是：

- Lot 1：**9**
- Lot 2–6：各 **10**
- Lot 7：**6**
- Lot 8：**10**
- Lot 9：**9**
- Lot 10：**4**

所以 Measurement coverage 並不是每個 Lot 都完整 10 片。

這時只能說「這些 Wafer 沒出現在 89-point CSV」，還不能說它們沒有 Process Data，也不能自己猜缺少的原因。後面會拿 `Process_data.nc` 直接比對。


## 4. 看 `Lot_status.xlsx` 的實驗結構

`Lot_status.xlsx` 不是只有一張乾淨的表，它同一個 sheet 裡其實放了兩段資訊：

1. 上半部是 10 個 Lot 的實驗摘要。
2. 下半部是 post-etch measurement 出現 NaN 的 Wafer 紀錄。

所以這裡先看原始版面，之後再分段整理。NaN 的逐片比對會留到 01。


### 4.1 先把整張 Excel 原樣讀進來

這裡用 `header=None`，因為同一張 sheet 裡有兩個不同格式的區塊，不能硬把第一列當成整張表唯一的 header。


In [10]:
lot_status_raw = pd.read_excel(
    lot_status_path,
    sheet_name="Tabelle1",
    header=None
)

print("Excel 原始維度：", lot_status_raw.shape)

lot_status_raw

Excel 原始維度： (36, 5)


,0,1,2,3,4
0,Lot No.,Date,Wafers,Measurements,Type
1,Lot 1,2th July 2024,10,89 & 9 points,3C
2,Lot 2,5th July 2024,10,89 & 9 points,1C
3,Lot 3,9th July 2024,10,89 & 9 points,9C
4,Lot 4,11th July 2024,10,89 & 9 points,3C - Si
5,Lot 5,19th July 2024,10,89 & 9 points,1C - Si
6,Lot 6,1st Aug 2024,10,89 & 9 points,9C - Si
7,Lot 7,5th Aug 2024,6,89 & 9 points,3C - SiO2
8,Lot 8,7th Aug 2024,10,89 points,3C - SiO2
9,Lot 9,21st Aug 2024,10,89 & 9 points,3C


**這個結果代表什麼？**

整張工作表是 **36 rows × 5 columns**。

從畫面可以看出，前面是 Lot 1–10 的摘要，到了大約第 15 列後開始變成 post-etch NaN 的紀錄。

所以後面一定要分段處理，不能把這 36 列直接當成同一張規則表格。


### 4.2 先取出 Lot 1–10 的摘要

這一格只抓前面的 Lot Summary，重新命名欄位，但先保留原始文字，不急著修日期或空白。

我想先確認資料本身記了哪些欄位，再把格式問題留到 01 清理。


In [11]:
lot_summary = lot_status_raw.iloc[1:11].copy()

lot_summary.columns = [
    "lot",
    "date",
    "wafers",
    "measurements",
    "conditioning",
]

lot_summary = lot_summary.reset_index(drop=True)

lot_summary

,lot,date,wafers,measurements,conditioning
0,Lot 1,2th July 2024,10,89 & 9 points,3C
1,Lot 2,5th July 2024,10,89 & 9 points,1C
2,Lot 3,9th July 2024,10,89 & 9 points,9C
3,Lot 4,11th July 2024,10,89 & 9 points,3C - Si
4,Lot 5,19th July 2024,10,89 & 9 points,1C - Si
5,Lot 6,1st Aug 2024,10,89 & 9 points,9C - Si
6,Lot 7,5th Aug 2024,6,89 & 9 points,3C - SiO2
7,Lot 8,7th Aug 2024,10,89 points,3C - SiO2
8,Lot 9,21st Aug 2024,10,89 & 9 points,3C
9,Lot 10,22nd Aug 2024,10,89 points,3C - SiO2


**這個結果代表什麼？**

現在成功取出 10 個 Lot 的：

- Date
- Wafer count
- Measurement type
- Conditioning

同時也能直接看到一些格式不一致，例如 `2th July 2024` 和多餘空白。這些不代表實驗內容錯了，但如果直接拿去 merge 或 groupby 會麻煩，所以 01 會統一整理。


### 4.3 把 `Lot 1` 這種文字轉成數字 Lot

Measurement Data 裡的 `lot_number` 是數字，所以這裡先把 `Lot 1`、`Lot 2` 這種文字抽成 1、2、3……，讓後面可以直接對資料。


In [12]:
#先只建立 numeric lot_number
lot_summary["lot_number"] = (
    lot_summary["lot"]
    .str.extract(r"(\d+)")
    .astype(int)
)

lot_summary[
    [
        "lot_number",
        "date",
        "wafers",
        "measurements",
        "conditioning",
    ]
]

,lot_number,date,wafers,measurements,conditioning
0,1,2th July 2024,10,89 & 9 points,3C
1,2,5th July 2024,10,89 & 9 points,1C
2,3,9th July 2024,10,89 & 9 points,9C
3,4,11th July 2024,10,89 & 9 points,3C - Si
4,5,19th July 2024,10,89 & 9 points,1C - Si
5,6,1st Aug 2024,10,89 & 9 points,9C - Si
6,7,5th Aug 2024,6,89 & 9 points,3C - SiO2
7,8,7th Aug 2024,10,89 points,3C - SiO2
8,9,21st Aug 2024,10,89 & 9 points,3C
9,10,22nd Aug 2024,10,89 points,3C - SiO2


**這個結果代表什麼？**

Lot 1–10 都成功轉成數值型 `lot_number`，而且原本的日期、Wafer 數、Measurement 和 Conditioning 內容都還保留。

這樣後面就可以用相同的 Lot 編號跟 Measurement Data 對齊。


### 4.4 Lot Status 一共記錄多少片 Wafer？

把 10 個 Lot 的 `wafers` 欄位加總，先確認整個實驗層級到底有多少片 Wafer。


In [13]:
#Process 實驗共有多少 Wafer？
total_process_wafers = int(lot_summary["wafers"].sum())

print("Lot Status 記錄的 Wafer 總數：", total_process_wafers)

Lot Status 記錄的 Wafer 總數： 96


**這個結果代表什麼？**

`Lot_status.xlsx` 加總後一共有 **96 片 Wafer**。

目前和 Measurement Data 對比是：

- Lot Status：**96 片**
- 89-point Measurement：**88 片**

中間差 **8 片**。

這時還不能直接說 8 片是「缺資料」；下一節會去看 `Process_data.nc` 有沒有這 8 片的 Process Data。


## 5. 看 `Process_data.nc` 是怎麼存的

接著換 Process Data。

這一段我主要想確認三件事：

1. NetCDF 裡到底有多少片 Wafer。
2. 一片 Wafer 的 Process Time Series 是什麼結構。
3. 不同 Wafer 的 Process Feature 欄位是不是一樣。

這裡還不做 Process QC，只先把資料結構搞懂。


### 5.1 NetCDF 裡有多少個 Wafer Group？

這個檔案是一個 Group 對應一片 Wafer，所以先數 Group 數量，再看前幾個 Group 名稱。


In [14]:
with Dataset(process_path, "r") as process_ds:
    process_groups = list(process_ds.groups.keys())

print("Process_data.nc 中的 Wafer group 數量：", len(process_groups))

print("\n前 10 個 group：")
for group_name in process_groups[:10]:
    print(group_name)

Process_data.nc 中的 Wafer group 數量： 96

前 10 個 group：
Day_2024_07_02_Wafer_01
Day_2024_07_02_Wafer_02
Day_2024_07_02_Wafer_03
Day_2024_07_02_Wafer_04
Day_2024_07_02_Wafer_05
Day_2024_07_02_Wafer_06
Day_2024_07_02_Wafer_07
Day_2024_07_02_Wafer_08
Day_2024_07_02_Wafer_09
Day_2024_07_02_Wafer_10


**這個結果代表什麼？**

`Process_data.nc` 一共有 **96 個 Wafer groups**，剛好和 `Lot_status.xlsx` 的 96 片完全一致。

Group 名稱像 `Day_2024_07_02_Wafer_01`，裡面已經含日期和 Wafer number，所以後面可以把它轉成和 Measurement Data 一樣的 `experiment_key`。


### 5.2 一片 Wafer Group 裡面放了什麼？

我先拿第一個 Group 當例子，看它有哪些 variables、每個 variable 的 shape 和 dtype。

目的是先弄清楚時間軸、Feature 名稱和 Process data matrix 是怎麼對上的。


In [15]:
#看看一片 wafer 裡面到底有什麼
with Dataset(process_path, "r") as process_ds:
    first_group_name = process_groups[0]
    first_group = process_ds.groups[first_group_name]

    print("第一個 Wafer Group：", first_group_name)

    print("\nGroup 內的 Variables：")
    for variable_name in first_group.variables:
        variable = first_group.variables[variable_name]
        print(
            f"{variable_name:<10} "
            f"shape={variable.shape}, "
            f"dtype={variable.dtype}"
        )

第一個 Wafer Group： Day_2024_07_02_Wafer_01

Group 內的 Variables：
times      shape=(3245,), dtype=float64
feature    shape=(44,), dtype=<class 'str'>
data       shape=(3245, 44), dtype=uint16


**這個結果代表什麼？**

第一片 Wafer 有：

- `times`：**3,245 個時間點**
- `feature`：**44 個欄位名稱**
- `data`：**3,245 × 44** 的編碼矩陣

所以 Process Data 不是「一片 Wafer 一列」，而是每片 Wafer 都有一段 **multivariate time series**。

不過第一片有 44 個 Features，不代表其他 95 片也一定是 44 個，後面還要逐片確認。


### 5.3 為什麼還要讀 `Dictionary_process.nc`？

`Process_data.nc` 的 `data` 不是直接存真實數值，而是存 dictionary index。

所以如果直接把那些 `uint16` 數字拿來分析會錯，必須先用 `Dictionary_process.nc` 解碼。


In [16]:
with Dataset(dictionary_path, "r") as dictionary_ds:
    decoder = dictionary_ds["data"][:]

print("Dictionary 長度：", len(decoder))
print("Dictionary 資料型態：", decoder.dtype)
print("前 10 個解碼值：")
print(decoder[:10])

Dictionary 長度： 49290
Dictionary 資料型態： float32
前 10 個解碼值：
[-103.57982   -97.628716  -95.91968   -93.53923   -90.76205   -85.268715
  -83.37657   -82.15583   -76.63198   -75.3502  ]


**這個結果代表什麼？**

Dictionary 一共有 **49,290 個 float32 解碼值**。

這表示 Process Data 裡的 `uint16` 真的只是 index，不是原始工程數值。後面所有 Process analysis 都必須先解碼，不能直接拿 index 算平均或相關。


### 5.4 實際解碼一片 Wafer 看看

用 dictionary 把第一片 Wafer 的 `data` 還原成實際數值，再確認解碼後的 shape 是否仍然和 `times`、`feature` 對得上。


In [17]:
#把第一片 wafer 解碼
with Dataset(dictionary_path, "r") as dictionary_ds:
    decoder = dictionary_ds["data"][:]

with Dataset(process_path, "r") as process_ds:
    first_group = process_ds.groups[first_group_name]

    encoded_data = first_group["data"][:]
    decoded_data = decoder[encoded_data]

    feature_names = first_group["feature"][:]
    times = first_group["times"][:]

print("Wafer：", first_group_name)
print("製程資料 shape：", decoded_data.shape)
print("時間點數量：", len(times))
print("Feature 數量：", len(feature_names))

Wafer： Day_2024_07_02_Wafer_01
製程資料 shape： (3245, 44)
時間點數量： 3245
Feature 數量： 44


**這個結果代表什麼？**

解碼後還是 **3,245 個時間點 × 44 個 Features**，而且時間點數量和 Feature 數量都和 Group metadata 一致。

所以解碼流程的維度沒有跑掉，可以繼續往下檢查這 44 個欄位是不是所有 Wafer 都有。


### 5.5 先看第一片 Wafer 的 Feature 名稱

這裡把第一片的 44 個 Feature 名稱全部列出來，只是先認識有哪些 signal。

我不會因為第一片有 44 個，就直接把 44 個當成整份資料的共同欄位。


In [18]:
#看 Feature 名稱
def clean_feature_name(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    return str(value)


first_feature_names = [
    clean_feature_name(value)
    for value in feature_names
]

for i, feature in enumerate(first_feature_names, start=1):
    print(f"{i:02d}. {feature}")

01. Stat3_Etch_MV_EpdIntensity
02. Stat3_Etch_MV_ForeLinePressure
03. Stat3_Etch_MV_Gas1Flow
04. Stat3_Etch_MV_Gas2Flow
05. Stat3_Etch_MV_Gas3Flow
06. Stat3_Etch_MV_Gas4Flow
07. Stat3_Etch_MV_Gas5Flow
08. Stat3_Etch_MV_Gas6Flow
09. Stat3_Etch_MV_Gas7Flow
10. Stat3_Etch_MV_Gas8Flow
11. Stat3_Etch_MV_Heater1Temp
12. Stat3_Etch_MV_Heater2Temp
13. Stat3_Etch_MV_Heater3Temp
14. Stat3_Etch_MV_Heater4Temp
15. Stat3_Etch_MV_Heater5Temp
16. Stat3_Etch_MV_Heater6Temp
17. Stat3_Etch_MV_Heater7Temp
18. Stat3_Etch_MV_Heater8Temp
19. Stat3_Etch_MV_HeliumBPFlow
20. Stat3_Etch_MV_HeliumBPPressure
21. Stat3_Etch_MV_PlatenDcBias
22. Stat3_Etch_MV_PlatenRFLoadCapacitor
23. Stat3_Etch_MV_PlatenRFLoadPower
24. Stat3_Etch_MV_PlatenRFPeakToPeak
25. Stat3_Etch_MV_PlatenRFReflectedPower
26. Stat3_Etch_MV_PlatenRFTuningCapacitor
27. Stat3_Etch_MV_Pressure
28. Stat3_Etch_MV_SourceRF2LoadCapacitor
29. Stat3_Etch_MV_SourceRF2LoadPower
30. Stat3_Etch_MV_SourceRF2PeakToPeak
31. Stat3_Etch_MV_SourceRF2ReflectedPower


**這個結果代表什麼？**

第一片確實有 44 個 Process Features，內容包含 gas flow、pressure、RF、temperature 和其他設備 signal。

這些欄位本身看起來很完整，但跨 Wafer 分析最重要的是「每片是不是都有」，所以接著要盤點全部 96 片的 Feature count。


## 6. 不同 Wafer 的 Process Feature 欄位一樣嗎？

如果不同 Wafer 有不同欄位，直接把它們拼在一起分析一定會出問題。

所以這一段先看全部 96 個 Group 的：

- Timepoint count
- Feature count

先確認資料到底有幾種欄位版本。


### 6.1 建立每片 Wafer 的結構表

逐一讀每個 Group 的 `n_timepoints` 和 `n_features`，先做成一張簡單的結構表。


In [19]:
#96 片 wafer 各有幾個 features？
process_structure_records = []

with Dataset(process_path, "r") as process_ds:
    for group_name, group in process_ds.groups.items():

        n_timepoints = len(group["times"][:])
        n_features = len(group["feature"][:])

        process_structure_records.append(
            {
                "group_name": group_name,
                "n_timepoints": n_timepoints,
                "n_features": n_features,
            }
        )

process_structure_df = pd.DataFrame(process_structure_records)

process_structure_df.head()

,group_name,n_timepoints,n_features
0,Day_2024_07_02_Wafer_01,3245,44
1,Day_2024_07_02_Wafer_02,3298,44
2,Day_2024_07_02_Wafer_03,3247,44
3,Day_2024_07_02_Wafer_04,3247,44
4,Day_2024_07_02_Wafer_05,3298,44


**這個結果代表什麼？**

表格前幾列都是 2024-07-02 的 Wafer，而且都顯示 44 個 Features。

但前幾列不能代表全部 96 片，所以還要看完整的 Feature count 分布。


### 6.2 全部 96 片的 Feature count 分布

直接統計每片 Wafer 有幾個 Process Features，看這份資料到底有幾種欄位數。


In [20]:
#統計 Feature 數量
process_structure_df["n_features"].value_counts().sort_index()

n_features
31    86
44    10
Name: count, dtype: int64

**這個結果代表什麼？**

96 片 Wafer 分成兩種欄位結構：

- **86 片：31 Features**
- **10 片：44 Features**

所以不能把 44 個欄位當成全資料共同欄位。後面一定要先搞清楚多出來的 13 個欄位是什麼、有没有真正的變化資訊。


### 6.3 哪 10 片是 44-Feature Wafer？

把 44-Feature Wafer 全部列出來，並把名單存下來，後面直接檢查這 10 片多出的 13 個欄位。


In [21]:
groups_with_44_features = (
    process_structure_df
    .loc[
        process_structure_df["n_features"] == 44,
        "group_name",
    ]
    .tolist()
)

process_structure_df.loc[
    process_structure_df["n_features"] == 44
]

,group_name,n_timepoints,n_features
0,Day_2024_07_02_Wafer_01,3245,44
1,Day_2024_07_02_Wafer_02,3298,44
2,Day_2024_07_02_Wafer_03,3247,44
3,Day_2024_07_02_Wafer_04,3247,44
4,Day_2024_07_02_Wafer_05,3298,44
5,Day_2024_07_02_Wafer_06,3246,44
6,Day_2024_07_02_Wafer_07,3247,44
7,Day_2024_07_02_Wafer_08,3298,44
8,Day_2024_07_02_Wafer_09,3246,44
9,Day_2024_07_02_Wafer_10,3247,44


**這個結果代表什麼？**

10 片 44-Feature Wafer 全部集中在 **2024-07-02**。

所以欄位差異不是零散發生在不同日期，而是集中在同一天。不過現在還不能說是設備設定、匯出版本或其他原因，只能先把這個結構事實記下來。


### 6.4 找一片 31-Feature Wafer 當比較樣本

接著從 86 片 31-Feature Wafer 裡拿第一片，和前面的 44-Feature Wafer 比較欄位名稱。


In [22]:
# 找出第一片只有 31 個 Process Features 的 Wafer

example_31_group_name = (
    process_structure_df
    .loc[
        process_structure_df["n_features"] == 31,
        "group_name"
    ]
    .iloc[0]
)

print("31 個 Feature 的範例 Wafer：")
print(example_31_group_name)

31 個 Feature 的範例 Wafer：
Day_2024_07_05_Wafer_01


**這個結果代表什麼？**

第一片 31-Feature Wafer 是 `Day_2024_07_05_Wafer_01`。

下一格只讀它的 31 個欄位名稱，拿來和第一片 44-Feature Wafer 做 set comparison。


### 6.5 看 31-Feature 版本有哪些欄位

把這片 Wafer 的 31 個 Feature 名稱列出來，確認 31-Feature 版本實際包含哪些 signals。


In [23]:
#讀取 31 欄版本名稱
# 讀取一片 31 個 Process Features 的 Wafer

with Dataset(process_path, "r") as process_ds:
    group_31 = process_ds.groups[example_31_group_name]

    feature_names_31 = [
        clean_feature_name(value)
        for value in group_31["feature"][:]
    ]

print("範例 Wafer：", example_31_group_name)
print("Feature 數量：", len(feature_names_31))

print("\n31 個 Process Features：")

for i, feature in enumerate(feature_names_31, start=1):
    print(f"{i:02d}. {feature}")

範例 Wafer： Day_2024_07_05_Wafer_01
Feature 數量： 31

31 個 Process Features：
01. Stat3_Etch_MV_EpdIntensity
02. Stat3_Etch_MV_ForeLinePressure
03. Stat3_Etch_MV_Gas1Flow
04. Stat3_Etch_MV_Gas2Flow
05. Stat3_Etch_MV_Gas3Flow
06. Stat3_Etch_MV_Gas4Flow
07. Stat3_Etch_MV_Gas5Flow
08. Stat3_Etch_MV_Gas7Flow
09. Stat3_Etch_MV_Gas8Flow
10. Stat3_Etch_MV_Heater1Temp
11. Stat3_Etch_MV_Heater2Temp
12. Stat3_Etch_MV_Heater3Temp
13. Stat3_Etch_MV_Heater4Temp
14. Stat3_Etch_MV_HeliumBPFlow
15. Stat3_Etch_MV_HeliumBPPressure
16. Stat3_Etch_MV_PlatenDcBias
17. Stat3_Etch_MV_PlatenRFLoadCapacitor
18. Stat3_Etch_MV_PlatenRFLoadPower
19. Stat3_Etch_MV_PlatenRFPeakToPeak
20. Stat3_Etch_MV_PlatenRFReflectedPower
21. Stat3_Etch_MV_PlatenRFTuningCapacitor
22. Stat3_Etch_MV_Pressure
23. Stat3_Etch_MV_SourceRF2LoadPower
24. Stat3_Etch_MV_SourceRF2PeakToPeak
25. Stat3_Etch_MV_SourceRF2ReflectedPower
26. Stat3_Etch_MV_SourceRF2TuningCapacitor
27. Stat3_Etch_MV_SourceRFLoadPower
28. Stat3_Etch_MV_SourceRFPeakToPeak

**這個結果代表什麼？**

這片 Wafer 確實只有 31 個 Features。

接著我會確認兩件事：

- 這 31 個是不是全部都包含在 44-Feature 版本裡。
- 44-Feature 版本到底多了哪 13 個欄位。


### 6.6 直接比較 31-Feature 和 44-Feature 兩種版本

把兩份 Feature name 轉成集合，直接算交集和差集，比只看欄位順序更乾淨。


In [24]:
#31 vs 44 直接比較
# 將兩種版本的欄位名稱轉成集合，方便比較

features_44 = set(first_feature_names)
features_31 = set(feature_names_31)

# 共同欄位
common_features = sorted(features_44 & features_31)

# 只有 44 欄版本才有的欄位
extra_features_44 = sorted(features_44 - features_31)

# 只有 31 欄版本才有的欄位
extra_features_31 = sorted(features_31 - features_44)

print("共同 Features 數量：", len(common_features))
print("44 欄版本額外 Features 數量：", len(extra_features_44))
print("31 欄版本額外 Features 數量：", len(extra_features_31))

共同 Features 數量： 31
44 欄版本額外 Features 數量： 13
31 欄版本額外 Features 數量： 0


**這個結果代表什麼？**

比對後可以看到：

- 共同 Features：**31**
- 只有 44-Feature 版本才有：**13**
- 只有 31-Feature 版本才有：**0**

所以 44-Feature 版本其實就是：

**共同的 31 個 Features + 額外 13 個 Features**

也就是說，31 個欄位不是另一套完全不同的資料格式，而是 44 欄版本裡本來就有的共同部分。這也表示後面如果要一次比較全部 96 片 Wafer，先以這 31 個共同 Features 為主會比較合理。

### 6.7 多出來的 13 個 Features 是哪些？

把 44-Feature 版本額外出現的 13 個欄位列出來，接著才能檢查它們是不是只是固定值，還是真的有製程變化。


In [25]:
#到底多出哪 13 個？
print("44 欄版本額外的 Process Features：\n")

for i, feature in enumerate(extra_features_44, start=1):
    print(f"{i:02d}. {feature}")

44 欄版本額外的 Process Features：

01. Stat3_Etch_MV_Gas6Flow
02. Stat3_Etch_MV_Heater5Temp
03. Stat3_Etch_MV_Heater6Temp
04. Stat3_Etch_MV_Heater7Temp
05. Stat3_Etch_MV_Heater8Temp
06. Stat3_Etch_MV_SourceRF2LoadCapacitor
07. Stat3_Etch_MV_SourceRFLoadCapacitor
08. Stat3_Etch_MV_ThermoCouple1Temp
09. Stat3_Etch_MV_ThermoCouple2Temp
10. Stat3_Etch_MV_ThermoCouple3Temp
11. Stat3_Etch_MV_ThermoCouple4Temp
12. Stat3_Etch_MV_attenuatorRatio
13. Stat3_Etch_MV_moriOuterCurrent


**這個結果代表什麼？**

多出的 13 個欄位主要包含 Gas6、Heater5–8、部分 capacitor、thermocouple、attenuator 和 outer current signals。

但光知道欄位名稱沒有意義。如果這些欄位全部都是固定 0 或固定 1，就算它們存在，也不能提供跨時間或跨 Wafer 的分析資訊。


### 6.8 不只看一片，直接檢查 10 片 × 13 個額外欄位

我不想只用第一片 Wafer 下結論，所以這裡把 10 片 44-Feature Wafer 的 13 個額外欄位全部檢查。

每個欄位都看：

- unique value count
- min
- max
- standard deviation

這樣才能知道它們在每片 Wafer 內到底有沒有變化。


In [26]:
#把 10 片的額外欄位全部檢查
extra_feature_check_records = []

with Dataset(dictionary_path, "r") as dictionary_ds:
    decoder = dictionary_ds["data"][:]

with Dataset(process_path, "r") as process_ds:

    for group_name in groups_with_44_features:

        group = process_ds.groups[group_name]

        encoded = group["data"][:]
        decoded = decoder[encoded]

        features = [
            clean_feature_name(value)
            for value in group["feature"][:]
        ]

        wafer_df = pd.DataFrame(
            decoded,
            columns=features
        )

        for feature in extra_features_44:

            extra_feature_check_records.append(
                {
                    "group_name": group_name,
                    "feature": feature,
                    "n_unique": wafer_df[feature].nunique(),
                    "min": wafer_df[feature].min(),
                    "max": wafer_df[feature].max(),
                    "std": wafer_df[feature].std(),
                }
            )

extra_feature_check_df = pd.DataFrame(
    extra_feature_check_records
)

extra_feature_check_df.head(20)

,group_name,feature,n_unique,min,max,std
0,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_Gas6Flow,1,0.0,0.0,0.0
1,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_Heater5Temp,1,0.0,0.0,0.0
2,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_Heater6Temp,1,0.0,0.0,0.0
3,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_Heater7Temp,1,0.0,0.0,0.0
4,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_Heater8Temp,1,0.0,0.0,0.0
5,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_SourceRF2LoadCapacitor,1,1.0,1.0,0.0
6,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_SourceRFLoadCapacitor,1,1.0,1.0,0.0
7,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_ThermoCouple1Temp,1,0.0,0.0,0.0
8,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_ThermoCouple2Temp,1,0.0,0.0,0.0
9,Day_2024_07_02_Wafer_01,Stat3_Etch_MV_ThermoCouple3Temp,1,0.0,0.0,0.0


**這個結果代表什麼？**

前 20 列已經可以看到很多額外欄位都是：

- `n_unique = 1`
- `std = 0`

看起來很像固定值。

不過這一格只顯示前 20 列，還不能只靠這些列就宣布 13 個欄位全部固定，所以下一步還要做 Feature-level summary。


### 6.9 再確認這些欄位在「不同 Wafer 之間」也沒有變

只看每片 Wafer 內 `std = 0` 還不夠。

例如某個欄位可能在 Wafer 1 全部是 1、Wafer 2 全部是 2，這樣每片內部都沒有變，但跨 Wafer 還是有資訊。

所以這裡對每個額外 Feature 再看：

- 最大 unique count
- 10 片合在一起的 min / max
- 最大 standard deviation

只有同時符合：

- `max_unique_values = 1`
- `max_std = 0`
- `min_value = max_value`

才真的可以說它在 10 片 Wafer 之間也完全固定。


In [27]:
# 將 13 個額外欄位彙整到 Feature level
extra_feature_overall_summary = (
    extra_feature_check_df
    .groupby("feature")
    .agg(
        max_unique_values=("n_unique", "max"),
        min_value=("min", "min"),
        max_value=("max", "max"),
        max_std=("std", "max"),
    )
    .sort_index()
)

extra_feature_overall_summary[
    "globally_constant_across_10_wafers"
] = (
    extra_feature_overall_summary["max_unique_values"].eq(1)
    & extra_feature_overall_summary["max_std"].eq(0)
    & extra_feature_overall_summary["min_value"].eq(
        extra_feature_overall_summary["max_value"]
    )
)

all_extra_features_globally_constant = (
    extra_feature_overall_summary[
        "globally_constant_across_10_wafers"
    ]
    .all()
)

print(
    "13 個額外 Features 是否在 10 片 Wafer 間也全部固定：",
    all_extra_features_globally_constant,
)

extra_feature_overall_summary


13 個額外 Features 是否在 10 片 Wafer 間也全部固定： True


,max_unique_values,min_value,max_value,max_std,globally_constant_across_10_wafers
feature,,,,,
Stat3_Etch_MV_Gas6Flow,1,0.0,0.0,0.0,True
Stat3_Etch_MV_Heater5Temp,1,0.0,0.0,0.0,True
Stat3_Etch_MV_Heater6Temp,1,0.0,0.0,0.0,True
Stat3_Etch_MV_Heater7Temp,1,0.0,0.0,0.0,True
Stat3_Etch_MV_Heater8Temp,1,0.0,0.0,0.0,True
Stat3_Etch_MV_SourceRF2LoadCapacitor,1,1.0,1.0,0.0,True
Stat3_Etch_MV_SourceRFLoadCapacitor,1,1.0,1.0,0.0,True
Stat3_Etch_MV_ThermoCouple1Temp,1,0.0,0.0,0.0,True
Stat3_Etch_MV_ThermoCouple2Temp,1,0.0,0.0,0.0,True


**這個結果代表什麼？**

這裡我同時看了「同一片 Wafer 裡會不會變」和「10 片 Wafer 之間會不會變」。

13 個額外 Features 都符合：

- 每片 Wafer 內只有 1 個 unique value
- 每片 Wafer 內 `std = 0`
- 10 片合在一起後也是 `min = max`

所以這 13 個欄位在 2024-07-02 的 10 片 Wafer 中其實都是**固定值**，沒有可以拿來比較 Wafer 差異的資訊。再加上另外 86 片根本沒有這 13 欄，所以後續跨全部 Wafer 的 Process 分析不使用它們是比較合理的。


### 6.10 最後直接算 96 片 Wafer 的 Feature 交集

前面用 31 和 44 的代表性 Wafer做了比較，最後我再直接把 **96 個 Group 的 Feature sets 全部取交集**。

這樣得到的才是真正「96 片每一片都有」的共同欄位。


In [28]:
# 收集 96 片 Wafer 的 Feature 集合

all_feature_sets = []

with Dataset(process_path, "r") as process_ds:

    for group_name, group in process_ds.groups.items():

        features = {
            clean_feature_name(value)
            for value in group["feature"][:]
        }

        all_feature_sets.append(features)

# 求所有 96 片 Wafer 的共同 Feature
common_features_all_wafers = set.intersection(
    *all_feature_sets
)

print(
    "所有 96 片 Wafer 共同擁有的 Process Features：",
    len(common_features_all_wafers)
)

所有 96 片 Wafer 共同擁有的 Process Features： 31


**這個結果代表什麼？**

96 片 Wafer 的 Feature 交集剛好是 **31 個**。

所以後面做 Process QC 或跨 Wafer 分析時，先從這 31 個共同 Features 出發是合理的。

至於這 31 個裡面是不是還有完全不變的欄位，00 先不處理，交給 01 做 Data Quality Check。


## 7. 把 Process Data 和 89-point Measurement 對起來

到這裡已經知道：

- Process Data：**96 片 Wafer**
- Measurement Data：**88 片 Wafer**

接著用統一的 `YYYY-MM-DD_XX` key 比對兩邊，直接分成：

- 同時有 Process + Measurement
- 只有 Process
- 只有 Measurement

這樣才知道後面需要兩種資料一起分析時，實際能用幾片 Wafer。


### 7.1 先把 NetCDF Group 名稱轉成 `experiment_key`

把像：

`Day_2024_07_02_Wafer_01`

轉成：

`2024-07-02_01`

這樣格式就能和 Measurement Data 的 `experiment_key` 一樣，後面才能精準比對。


In [29]:
#從 Process group 名稱拆日期和 wafer number
import re
process_wafer_records = []

for group_name in process_groups:

    match = re.match(
        r"Day_(\d{4}_\d{2}_\d{2})_Wafer_(\d+)",
        group_name
    )

    if match:

        date_text = match.group(1)
        wafer_number = int(match.group(2))

        experiment_key = (
            date_text.replace("_", "-")
            + "_"
            + f"{wafer_number:02d}"
        )

        process_wafer_records.append(
            {
                "group_name": group_name,
                "experiment_key": experiment_key,
                "wafer_number": wafer_number,
            }
        )

process_wafer_df = pd.DataFrame(
    process_wafer_records
)

process_wafer_df.head()

,group_name,experiment_key,wafer_number
0,Day_2024_07_02_Wafer_01,2024-07-02_01,1
1,Day_2024_07_02_Wafer_02,2024-07-02_02,2
2,Day_2024_07_02_Wafer_03,2024-07-02_03,3
3,Day_2024_07_02_Wafer_04,2024-07-02_04,4
4,Day_2024_07_02_Wafer_05,2024-07-02_05,5


**這個結果代表什麼？**

轉換結果的格式正確，例如：

`Day_2024_07_02_Wafer_01 → 2024-07-02_01`

所以 Process Data 現在有一個可以直接和 Measurement Data 對齊的 Wafer-level key。


### 7.2 建立 Measurement Wafer 名單

Measurement Data 有 7,832 列，但那是 point-level records。

這裡只保留 unique Wafer keys，避免拿 7,832 個點去和 96 個 Process groups 比較。


In [30]:
#建立 Measurement Wafer 名單
measurement_wafers = (
    measurement_df[
        [
            "experiment_key",
            "lot_number",
            "wafer_number",
        ]
    ]
    .drop_duplicates()
    .sort_values("experiment_key")
    .reset_index(drop=True)
)

print(
    "Measurement Wafer 數量：",
    len(measurement_wafers)
)

measurement_wafers.head()

Measurement Wafer 數量： 88


,experiment_key,lot_number,wafer_number
0,2024-07-02_01,1,1
1,2024-07-02_02,1,2
2,2024-07-02_03,1,3
3,2024-07-02_04,1,4
4,2024-07-02_05,1,5


**這個結果代表什麼？**

Measurement Wafer 名單一共是 **88 片**，和前面用 `experiment_key` 算到的數字完全一致。

所以 Wafer-level 名單沒有因為去重或整理而少掉或多出資料。


### 7.3 正式比對兩邊的 Wafer 名單

用集合的交集和差集，直接看三種情況：

- 兩邊都有
- 只有 Process
- 只有 Measurement


In [31]:
#正式找缺的 8 片
process_keys = set(
    process_wafer_df["experiment_key"]
)

measurement_keys = set(
    measurement_wafers["experiment_key"]
)

only_process = sorted(
    process_keys - measurement_keys
)

only_measurement = sorted(
    measurement_keys - process_keys
)

both = sorted(
    process_keys & measurement_keys
)

print("同時有 Process + Measurement：", len(both))
print("只有 Process：", len(only_process))
print("只有 Measurement：", len(only_measurement))

print("\n只有 Process、沒有 89-point Measurement 的 Wafer：")

for key in only_process:
    print("-", key)

同時有 Process + Measurement： 88
只有 Process： 8
只有 Measurement： 0

只有 Process、沒有 89-point Measurement 的 Wafer：
- 2024-07-02_07
- 2024-08-21_09
- 2024-08-22_05
- 2024-08-22_06
- 2024-08-22_07
- 2024-08-22_08
- 2024-08-22_09
- 2024-08-22_10


**這個結果代表什麼？**

比對後得到：

- **88 片**：同時有 Process + Measurement
- **8 片**：只有 Process
- **0 片**：只有 Measurement

所以只要後面的分析同時需要 Process 和最終 Measurement，最大完整樣本數就是 **88 片 Wafer**。

那 8 片 Process-only Wafer 也已經列出來。現在資料只告訴我它們缺 89-point Measurement，沒有足夠證據說明為什麼缺，所以這裡不自己猜原因。


## 8. 到這裡可以整理出整份資料的階層

### Lot Level
整個實驗有 **10 個 Lots**。Lot 是後面看 Conditioning、批次差異和 wafer sequence 時的重要分組。

### Wafer Level
`Process_data.nc` 一共有 **96 片 Process Wafers**：

- 88 片：Process + 89-point Measurement
- 8 片：只有 Process
- 0 片：只有 Measurement

資料集說明提到 Lot 7 最後 4 片 Wafer 因設備問題受損，所以 Lot 7 的 Process Data 只有 6 片。其他 Process-only 差異則只按照實際 key comparison 記錄，不另外猜原因。

### Measurement-Point Level
有 Measurement 的 88 片 Wafer，每片目前都確認有 **89 筆 records**：

**88 × 89 = 7,832 records**

00 這裡只證明每片有 89 列。X/Y 是否真的形成 89 個 unique spatial positions，留到 01 用 duplicate key 和 spatial-grid check 再確認。

後面做模型時，基本分析單位會是一片 Wafer，但 Wafer 仍然被包在不同 Lot 裡，所以驗證方式一定要保留 Lot grouping。


### 8.1 把關鍵數字整理成一張表

前面已經分開驗證了很多數量，這裡把它們集中在一張 summary table，之後 01 可以直接拿來對照，不用重新猜資料規模。


In [32]:
# 建立資料結構摘要表
data_structure_summary = pd.DataFrame(
    {
        "項目": [
            "Lot 數量",
            "Process Wafer 數量",
            "有 89-point Measurement 的 Wafer 數量",
            "同時有 Process + Measurement 的 Wafer 數量",
            "只有 Process 的 Wafer 數量",
            "只有 Measurement 的 Wafer 數量",
            "每片 Measured Wafer 的 Measurement Records",
            "89-point Measurement 總列數",
            "所有 Wafer 共同 Process Features",
            "2024-07-02 額外 Process Features",
        ],
        "數量": [
            measurement_df["lot_number"].nunique(),
            len(process_wafer_df),
            len(measurement_wafers),
            len(both),
            len(only_process),
            len(only_measurement),
            int(records_per_wafer["measurement_records"].min()),
            len(measurement_df),
            len(common_features_all_wafers),
            len(extra_features_44),
        ],
    }
)

data_structure_summary


,項目,數量
0,Lot 數量,10
1,Process Wafer 數量,96
2,有 89-point Measurement 的 Wafer 數量,88
3,同時有 Process + Measurement 的 Wafer 數量,88
4,只有 Process 的 Wafer 數量,8
5,只有 Measurement 的 Wafer 數量,0
6,每片 Measured Wafer 的 Measurement Records,89
7,89-point Measurement 總列數,7832
8,所有 Wafer 共同 Process Features,31
9,2024-07-02 額外 Process Features,13


**這個結果代表什麼？**

把前面的數字放在一起後，彼此是對得上的：

- **10 Lots**
- **96 Process Wafers**
- **88 Measured Wafers**
- 每片 **89 Measurement Records**
- 總共 **7,832 Measurement Records**
- **31 個共同 Process Features**

這代表 Lot、Wafer、Measurement Record 和 Process Feature 的主要結構目前是一致的。

唯一還沒有在 00 宣告的是「89 個 unique X/Y positions」，因為那要等 01 的 spatial key 驗證。


### 8.2 用最簡單的方式畫出資料階層

```text
整個實驗
│
├── 10 個 Lot
│   │
│   └── 共 96 片有 Process Data 的 Wafer
│       ├── 88 片：Process + 89-point Measurement
│       └──  8 片：Process only
│
└── 88 片有 89-point Measurement 的 Wafer
    └── 每片 89 筆 Measurement Records
        └── 共 7,832 筆 Measurement Records

※ 89 筆 records 是否就是 89 個 unique X/Y positions，01 再驗證。
```


## 9. 最後做一次結構一致性檢查

在結束 00 之前，我把前面最重要的關係都寫成 boolean checks。

不是只看數量，而是連 key 一對一、Process / Measurement 對齊、31/44 Feature 結構，以及 13 個額外欄位是否固定都一起檢查。

這樣如果未來 Raw Data 或前面程式被改過，只要其中一個條件不再成立，就能馬上發現。


In [33]:
# 資料結構一致性檢查
checks = {
    "Process Data 應共有 96 片 Wafer":
        len(process_wafer_df) == 96,

    "Measurement Data 應共有 88 片 Wafer":
        len(measurement_wafers) == 88,

    "experiment_key 應一對一對應 Lot/Wafer":
        experiment_key_to_one_wafer,

    "Lot/Wafer 應一對一對應 experiment_key":
        wafer_to_one_experiment_key,

    "88 片 Wafer 每片都應有 89 筆 Measurement Records":
        all_have_89_records,

    "88 × 89 應等於 Measurement Data 總列數":
        len(measurement_wafers) * 89 == len(measurement_df),

    "Process + Measurement 共同 Wafer 應為 88 片":
        len(both) == 88,

    "只有 Process 的 Wafer 應為 8 片":
        len(only_process) == 8,

    "不應存在只有 Measurement、沒有 Process 的 Wafer":
        len(only_measurement) == 0,

    "所有 96 片 Wafer 共同 Process Features 應為 31 個":
        len(common_features_all_wafers) == 31,

    "44 欄版本應額外多出 13 個 Features":
        len(extra_features_44) == 13,

    "額外 13 個 Features 在 10 片 44-Feature Wafers 間應全部固定":
        all_extra_features_globally_constant,
}

check_results = pd.DataFrame(
    {
        "檢查項目": checks.keys(),
        "是否通過": checks.values(),
    }
)

check_results


,檢查項目,是否通過
0,Process Data 應共有 96 片 Wafer,True
1,Measurement Data 應共有 88 片 Wafer,True
2,experiment_key 應一對一對應 Lot/Wafer,True
3,Lot/Wafer 應一對一對應 experiment_key,True
4,88 片 Wafer 每片都應有 89 筆 Measurement Records,True
5,88 × 89 應等於 Measurement Data 總列數,True
6,Process + Measurement 共同 Wafer 應為 88 片,True
7,只有 Process 的 Wafer 應為 8 片,True
8,不應存在只有 Measurement、沒有 Process 的 Wafer,True
9,所有 96 片 Wafer 共同 Process Features 應為 31 個,True


**這個結果代表什麼？**

這張檢查表的 12 個條件全部都是 `True`。

也就是目前可以同時確認：

- Process Wafers = **96**
- Measurement Wafers = **88**
- `experiment_key ↔ Lot/Wafer` 一對一
- 每片 = **89 records**
- **88 × 89 = 7,832**
- Process + Measurement = **88**
- Process only = **8**
- Measurement only = **0**
- 全部 96 片共同 Process Features = **31**
- 44-Feature 版本多 **13** 個欄位
- 這 13 個額外欄位在那 10 片 Wafer 間全部固定

所以 00 的主要資料結構目前沒有互相矛盾。

不過這裡仍然只確認每片 89 筆 records；X/Y 是否 89 個 unique positions，還是等 01 做。


最後再用 `assert` 把剛才的檢查表變成硬性條件。

這樣未來如果 Raw Data 或前面程式被修改，只要其中任何一項變成 `False`，Notebook 就會直接停下來，不會把結構已經不一致的資料繼續傳到 01。


In [34]:
all_checks_passed = check_results["是否通過"].all()

assert all_checks_passed, (
    "Data structure validation failed. "
    "Please review the checks above before continuing."
)

print("所有資料結構檢查通過。")


所有資料結構檢查通過。


**這個結果代表什麼？**

`assert` 沒有報錯，而且顯示「所有資料結構檢查通過」。

這代表目前 00 裡設定的所有結構條件都成立，可以把這些結果當成 01 的起點。

如果未來其中任何條件不成立，這一格就會直接報錯，避免後面的分析默默接著跑錯資料。


## 10. 這份 Notebook 的整體結論

做到這裡，我已經把後面分析最容易搞混的資料結構先釐清了。

整個實驗一共有 **10 個 Lots、96 片有 Process Data 的 Wafers**。89-point CSV 裡則有 **88 片有 Measurement 的 Wafers**，而且每片剛好都有 **89 筆 records**，所以總列數就是 **88 × 89 = 7,832**。`experiment_key` 和 `lot_number + wafer_number` 也確認是一對一，所以後面可以用 `experiment_key` 當作跨檔案對齊 Wafer 的 key。

Process Data 和 Measurement 對齊後，結果是 **88 片同時有 Process + Measurement、8 片只有 Process、0 片只有 Measurement**。這代表之後只要分析同時需要製程參數和最終量測結果，最多就是 88 片 Wafer。

Process 欄位也不是每片都完全一樣。96 片裡有 **86 片是 31 個 Features，10 片是 44 個 Features**。多出來的 13 個欄位全部集中在 2024-07-02 的 10 片 Wafer，而且我確認它們在這 10 片裡從頭到尾都是固定值，沒有跨 Wafer 的變化資訊。所以後面如果要一次比較全部 Wafer，最合理的起點就是 **31 個共同 Features**，而不是硬把 44 個欄位全部放進去。

這一份也讓後面的模型限制很清楚：真正可以拿來做有品質標籤模型的樣本最多只有 **88 片 Wafer**，而且這 88 片又分在 10 個 Lots 裡，不能把它們當成 88 個彼此完全獨立的樣本。

所以 00 的重點不是判斷製程好不好，而是先確認：**資料有幾層、Wafer 要怎麼對齊、Process 和 Measurement 哪些片有重疊，以及哪些 Process Features 可以跨全部 Wafer 比較。** 這些先確認清楚後，01 才能繼續檢查缺失值、量測品質和 Process 時間序列有哪些問題需要處理。

## 11. 下一步

下一份是：

**`01_data_quality.ipynb`｜資料品質檢查與分析資料準備**

00 已經回答「資料長什麼樣、怎麼對得起來」，01 就接著處理真正的資料品質問題，包括 missing / interpolation、空間座標、Lot metadata、Process NaN / Inf、constant features 和 timestamp gap。

也就是從「先把資料看懂」，進到「確認哪些資料可以放心拿去分析」。
